# 05 — Gold | Pre-Game Features for the Win-Probability Model

**Project:** MLB Performance Intelligence Platform

One row per game (completed **and** upcoming) with what was known **before first pitch**, from the
home team's point of view. Strict point-in-time logic: every number comes from games played before
that game (an "as-of" join), so the model cannot see the result it is trying to predict.

| Factor | Feature (home − away unless noted) | How it is computed |
|---|---|---|
| Team strength | `d_win_pct`, `d_run_diff` | Season-to-date win % and run differential per game, regressed toward last season (20 games of prior weight) |
| Recent form | `d_last10` | Last-10 win %, regressed toward the season level |
| Home / away split | `d_split` | Home team's home win % vs away team's road win % |
| Starting pitcher | `d_starter_fip`, `d_starter_kbb` | Starter's FIP and K−BB% before the game (season-to-date, regressed toward last season); actual starter for played games, **probable starter** for upcoming ones |
| Bullpen | `d_bullpen_era`, `d_bullpen_fatigue` | Relievers' ERA to date and outs thrown in the previous 3 days |
| Rest | `d_rest` | Days since each team's previous game (capped at 5) |
| Ballpark | `park_factor` | Runs per game at the venue vs league, from **previous seasons only**, regressed toward 1.00 |
| Context | `is_postseason` | Postseason game flag |

Label: `home_win` (1/0) for completed games; NULL for upcoming games, which the model scores.

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
for _m in (C, P):
    importlib.reload(_m)

RUN_ID = P.new_run_id()
T = lambda s, n: C.table(s, n)

# Priors (league-typical values) used when a team or pitcher has little or no history
K_TEAM, K_SPLIT, K_FIP_IP, K_KBB_BF, K_PEN_OUTS, K_PARK_GAMES = 20, 15, 40, 150, 150, 100
LG_FIP, LG_KBB, LG_PEN_ERA, FIP_CONST = 4.30, 12.0, 4.10, 3.15

## Team state before each game (as-of join)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tg AS
SELECT tg.game_pk, tg.team_id, tg.opponent_id, tg.is_home, tg.season, tg.season_type, tg.official_date,
       g.game_datetime_utc AS t, tg.is_completed, tg.is_win, tg.runs_scored, tg.runs_allowed
FROM {T('silver', 'fact_team_game')} tg JOIN {T('silver', 'fact_game')} g USING (game_pk)
WHERE tg.is_completed OR g.status_code NOT IN ('D', 'C')   -- drop postponed/cancelled leftovers
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW team_state AS
WITH states AS (
  SELECT team_id, season, t, game_pk, 1 AS kind,
    COUNT(*) OVER w AS g, SUM(INT(is_win)) OVER w AS w, SUM(runs_scored) OVER w AS rs, SUM(runs_allowed) OVER w AS ra,
    SUM(INT(is_win)) OVER w10 AS w10, COUNT(*) OVER w10 AS g10,
    SUM(INT(is_home)) OVER w AS hg, SUM(INT(is_home AND is_win)) OVER w AS hw,
    SUM(INT(NOT is_home)) OVER w AS ag, SUM(INT(NOT is_home AND is_win)) OVER w AS aw,
    official_date AS last_game_date
  FROM tg WHERE is_completed
  WINDOW w AS (PARTITION BY team_id, season ORDER BY t, game_pk ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
         w10 AS (PARTITION BY team_id, season ORDER BY t, game_pk ROWS BETWEEN 9 PRECEDING AND CURRENT ROW)
),
u AS (
  SELECT team_id, season, t, game_pk, 0 AS kind, CAST(NULL AS BIGINT) g, CAST(NULL AS BIGINT) w, CAST(NULL AS BIGINT) rs,
         CAST(NULL AS BIGINT) ra, CAST(NULL AS BIGINT) w10, CAST(NULL AS BIGINT) g10, CAST(NULL AS BIGINT) hg,
         CAST(NULL AS BIGINT) hw, CAST(NULL AS BIGINT) ag, CAST(NULL AS BIGINT) aw, CAST(NULL AS DATE) last_game_date
  FROM tg
  UNION ALL SELECT * FROM states
),
asof AS (
  SELECT team_id, season, game_pk, kind,
    LAST(g, TRUE) OVER x AS g, LAST(w, TRUE) OVER x AS w, LAST(rs, TRUE) OVER x AS rs, LAST(ra, TRUE) OVER x AS ra,
    LAST(w10, TRUE) OVER x AS w10, LAST(g10, TRUE) OVER x AS g10, LAST(hg, TRUE) OVER x AS hg, LAST(hw, TRUE) OVER x AS hw,
    LAST(ag, TRUE) OVER x AS ag, LAST(aw, TRUE) OVER x AS aw, LAST(last_game_date, TRUE) OVER x AS last_game_date
  FROM u
  WINDOW x AS (PARTITION BY team_id, season ORDER BY t, kind, game_pk ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
)
SELECT team_id, season, game_pk, COALESCE(g, 0) g, COALESCE(w, 0) w, COALESCE(rs, 0) rs, COALESCE(ra, 0) ra,
       COALESCE(w10, 0) w10, COALESCE(g10, 0) g10, COALESCE(hg, 0) hg, COALESCE(hw, 0) hw, COALESCE(ag, 0) ag,
       COALESCE(aw, 0) aw, last_game_date
FROM asof WHERE kind = 0
""")

## Starting pitcher before each game

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW starter AS
WITH actual AS (
  SELECT game_pk, team_id, MIN(player_id) AS player_id FROM {T('silver', 'fact_pitching_game')} WHERE is_starter GROUP BY ALL
),
probable AS (
  SELECT game_pk, home_team_id AS team_id, home_probable_pitcher_id AS player_id FROM {T('silver', 'fact_game')}
  UNION ALL SELECT game_pk, away_team_id, away_probable_pitcher_id FROM {T('silver', 'fact_game')}
),
pick AS (
  SELECT tg.game_pk, tg.team_id, tg.season, tg.official_date,
         CASE WHEN tg.is_completed THEN COALESCE(a.player_id, p.player_id) ELSE p.player_id END AS starter_id
  FROM tg LEFT JOIN actual a USING (game_pk, team_id) LEFT JOIN probable p USING (game_pk, team_id)
),
hist AS (
  SELECT k.game_pk, k.team_id, SUM(x.outs) outs, SUM(x.er) er, SUM(x.so) so, SUM(x.bb) bb, SUM(x.hbp) hbp,
         SUM(x.hr) hr, SUM(x.batters_faced) bf
  FROM pick k JOIN {T('silver', 'fact_pitching_game')} x
    ON x.player_id = k.starter_id AND x.season = k.season AND x.official_date < k.official_date
  GROUP BY ALL
)
SELECT k.game_pk, k.team_id, k.starter_id, k.starter_id IS NOT NULL AS starter_known,
  COALESCE(h.outs, 0) AS sp_outs,
  (13 * h.hr + 3 * (h.bb + h.hbp) - 2 * h.so) / NULLIF(h.outs / 3, 0) + {FIP_CONST} AS sp_fip_std,
  100 * (h.so - h.bb) / NULLIF(h.bf, 0) AS sp_kbb_std, COALESCE(h.bf, 0) AS sp_bf,
  ps.fip AS prev_fip, ps.outs AS prev_outs, ps.k_minus_bb_pct AS prev_kbb
FROM pick k
LEFT JOIN hist h USING (game_pk, team_id)
LEFT JOIN {T('gold', 'player_season_pitching')} ps
  ON ps.player_id = k.starter_id AND ps.season = k.season - 1 AND ps.season_type = 'Regular Season'
""")

## Bullpen quality and fatigue

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW bullpen AS
WITH daily AS (
  SELECT team_id, season, official_date AS d, SUM(outs) outs, SUM(er) er
  FROM {T('silver', 'fact_pitching_game')} WHERE NOT is_starter GROUP BY ALL
),
cum AS (
  SELECT team_id, season, d, 1 AS kind,
         SUM(outs) OVER w AS c_outs, SUM(er) OVER w AS c_er
  FROM daily WINDOW w AS (PARTITION BY team_id, season ORDER BY d ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
),
u AS (
  SELECT team_id, season, official_date AS d, 0 AS kind, game_pk, CAST(NULL AS BIGINT) c_outs, CAST(NULL AS BIGINT) c_er FROM tg
  UNION ALL SELECT team_id, season, d, kind, CAST(NULL AS BIGINT), c_outs, c_er FROM cum
),
asof AS (
  SELECT team_id, game_pk, kind, LAST(c_outs, TRUE) OVER x AS c_outs, LAST(c_er, TRUE) OVER x AS c_er
  FROM u WINDOW x AS (PARTITION BY team_id, season ORDER BY d, kind ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
),
fatigue AS (
  SELECT tg.game_pk, tg.team_id, SUM(dl.outs) AS outs_3d
  FROM tg JOIN daily dl ON dl.team_id = tg.team_id AND dl.d BETWEEN DATE_SUB(tg.official_date, 3) AND DATE_SUB(tg.official_date, 1)
  GROUP BY ALL
)
SELECT a.game_pk, a.team_id,
  (27 * COALESCE(a.c_er, 0) + {LG_PEN_ERA} * {K_PEN_OUTS}) / (COALESCE(a.c_outs, 0) + {K_PEN_OUTS}) AS bp_era,
  COALESCE(f.outs_3d, 0) AS bp_outs_3d
FROM asof a LEFT JOIN fatigue f USING (game_pk, team_id)
WHERE a.kind = 0
""")

## Park factor (previous seasons only)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW park AS
WITH vs AS (
  SELECT venue_id, season, SUM(total_runs) runs, COUNT(*) games
  FROM {T('silver', 'fact_game')} WHERE is_completed AND game_type = 'R' GROUP BY ALL
),
lg AS (SELECT season, SUM(runs) / NULLIF(SUM(games), 0) AS lg_rpg FROM vs GROUP BY season),
seasons AS (SELECT DISTINCT season FROM tg),
prior AS (
  SELECT s.season, v.venue_id, SUM(v.runs) runs, SUM(v.games) games, SUM(v.games * l.lg_rpg) exp_runs
  FROM seasons s JOIN vs v ON v.season < s.season JOIN lg l ON l.season = v.season
  GROUP BY ALL
)
SELECT season, venue_id, games AS park_games,
       (games * (runs / NULLIF(exp_runs, 0)) + {K_PARK_GAMES} * 1.0) / (games + {K_PARK_GAMES}) AS park_factor
FROM prior
""")

## Assemble one row per game

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW team_features AS
SELECT tg.game_pk, tg.team_id, tg.is_home, tg.season, tg.official_date, tg.is_completed, tg.is_win,
  s.g AS games_before,
  0.5 + 0.7 * (COALESCE(prev.win_pct, 0.5) - 0.5) AS prior_wp,
  (s.w + {K_TEAM} * (0.5 + 0.7 * (COALESCE(prev.win_pct, 0.5) - 0.5))) / (s.g + {K_TEAM}) AS wp_reg,
  ((s.rs - s.ra) + {K_TEAM} * 0.7 * COALESCE(prev.run_differential / prev.games, 0)) / (s.g + {K_TEAM}) AS rdpg_reg,
  LEAST(COALESCE(DATEDIFF(tg.official_date, s.last_game_date), 5), 5) AS rest_days,
  s.w10, s.g10, s.hw, s.hg, s.aw, s.ag,
  st.starter_id, st.starter_known,
  CASE WHEN st.starter_known THEN
    (COALESCE(st.sp_fip_std, 0) * st.sp_outs / 3
     + {K_FIP_IP} * COALESCE((st.prev_fip * st.prev_outs / 3 + {LG_FIP} * 30) / (st.prev_outs / 3 + 30), {LG_FIP} + 0.15))
    / (st.sp_outs / 3 + {K_FIP_IP})
  ELSE {LG_FIP} END AS sp_fip,
  CASE WHEN st.starter_known THEN
    (COALESCE(st.sp_kbb_std, 0) * st.sp_bf + {K_KBB_BF} * COALESCE(st.prev_kbb, {LG_KBB} - 1)) / (st.sp_bf + {K_KBB_BF})
  ELSE {LG_KBB} END AS sp_kbb,
  st.sp_outs, bp.bp_era, bp.bp_outs_3d
FROM tg
JOIN team_state s USING (game_pk, team_id)
LEFT JOIN starter st USING (game_pk, team_id)
LEFT JOIN bullpen bp USING (game_pk, team_id)
LEFT JOIN {T('gold', 'team_season_summary')} prev
  ON prev.team_id = tg.team_id AND prev.season = tg.season - 1 AND prev.season_type = 'Regular Season'
""")

features = spark.sql(f"""
WITH tf AS (
  SELECT *, (w10 + 5 * wp_reg) / (g10 + 5) AS last10_reg,
         CASE WHEN is_home THEN (hw + {K_SPLIT} * wp_reg) / (hg + {K_SPLIT})
              ELSE (aw + {K_SPLIT} * wp_reg) / (ag + {K_SPLIT}) END AS split_reg
  FROM team_features
)
SELECT g.game_pk, g.season, g.season_type, g.game_type, g.official_date, g.game_datetime_utc, g.status_detail,
  g.is_completed, g.home_team_id, g.away_team_id, g.venue_id, g.venue_name,
  CASE WHEN g.is_completed AND NOT g.is_tie THEN INT(g.home_score > g.away_score) END AS home_win,
  h.starter_id AS home_starter_id, a.starter_id AS away_starter_id,
  h.starter_known AS home_starter_known, a.starter_known AS away_starter_known,
  h.games_before AS home_games_before, a.games_before AS away_games_before,
  h.wp_reg AS home_wp, a.wp_reg AS away_wp, h.rdpg_reg AS home_rdpg, a.rdpg_reg AS away_rdpg,
  h.last10_reg AS home_last10, a.last10_reg AS away_last10, h.split_reg AS home_split, a.split_reg AS away_split,
  h.sp_fip AS home_sp_fip, a.sp_fip AS away_sp_fip, h.sp_kbb AS home_sp_kbb, a.sp_kbb AS away_sp_kbb,
  h.bp_era AS home_bp_era, a.bp_era AS away_bp_era, h.bp_outs_3d AS home_bp_outs_3d, a.bp_outs_3d AS away_bp_outs_3d,
  h.rest_days AS home_rest, a.rest_days AS away_rest,
  -- model features: positive values favour the home team
  h.wp_reg - a.wp_reg AS d_win_pct,
  h.rdpg_reg - a.rdpg_reg AS d_run_diff,
  h.last10_reg - a.last10_reg AS d_last10,
  h.split_reg - a.split_reg AS d_split,
  a.sp_fip - h.sp_fip AS d_starter_fip,
  h.sp_kbb - a.sp_kbb AS d_starter_kbb,
  a.bp_era - h.bp_era AS d_bullpen_era,
  a.bp_outs_3d - h.bp_outs_3d AS d_bullpen_fatigue,
  h.rest_days - a.rest_days AS d_rest,
  COALESCE(pk.park_factor, 1.0) AS park_factor,
  INT(g.season_type = 'Postseason') AS is_postseason
FROM {T('silver', 'fact_game')} g
JOIN tf h ON h.game_pk = g.game_pk AND h.is_home
JOIN tf a ON a.game_pk = g.game_pk AND NOT a.is_home
LEFT JOIN park pk ON pk.season = g.season AND pk.venue_id = g.venue_id
""")

with P.log_step(spark, RUN_ID, "gold", "game_features") as step:
    step.records = P.replace_table(spark, features, T("gold", "game_features"),
        "Gold: one row per game with point-in-time pre-game features (home minus away) for the win-probability model")

In [ ]:
display(spark.sql(f"""
    SELECT season, season_type, COUNT(*) AS games, SUM(INT(is_completed)) AS completed,
           SUM(INT(NOT is_completed)) AS upcoming, ROUND(AVG(home_win), 3) AS home_win_rate,
           ROUND(AVG(INT(home_starter_known AND away_starter_known)), 3) AS starters_known
    FROM {T('gold', 'game_features')} GROUP BY ALL ORDER BY season, season_type"""))